# UppASD Interactive Setup

First-contact notebook: build a system, run UppASD, inspect results.

In [ ]:
import importlib.util
import numpy as np

if importlib.util.find_spec("uppasd") is None:
    print("Installing uppasd…")
    !pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple uppasd

from uppasd.core.system import SpinSystem
from uppasd.core.exchange import ExchangeShellTable
from uppasd.input.inputdata import ASDInput
from uppasd.run.simulator import ASDWorkspace, UppASDSimulator
from uppasd.core.results import ASDResults
from uppasd.viz.trajectory import plot_3d_trajectory
from uppasd.viz.io import spin_snapshots
from uppasd.viz.spectrogram import plot_xt_map


## Define a simple 1D Heisenberg chain

In [ ]:
N = 1
a = 1.0
cell = np.diag([N*a, a, a])
positions = np.zeros((N,3))
positions[:,0] = np.arange(N)*a
species = np.ones(N, dtype=int)
moments = np.zeros((N,3))
moments[:,2] = 1.0
system = SpinSystem(cell, positions, species, moments)


## Exchange interactions

In [ ]:
exchange = ExchangeShellTable()
J = 1.0
for i in range(1, N):
    exchange.add_bond(i, i+1, 1, [a,0,0], J)
    exchange.add_bond(i+1, i, 1, [-a,0,0], J)


## Input and run

In [ ]:
inp = ASDInput()
inp.block("system").set(
    simid="chain",
    ncell=(100, 1, 1), # 100 atom long chain in x
    bc=(0, 0, 0),          # vacuum in all directions
    cell=cell,
    do_prnstruct=2,
)
inp.block('dynamics').set(mode='S', 
                          temp=10.0, 
                          nstep=50000, 
                          damping=0.05)
inp.block('measure').set(do_tottraj='Y',
                        tottraj_step=100,
                         ntraj=(1,'\n',1,100,10),
                        )

workdir = 'interactive_run'
ws = ASDWorkspace(workdir, clean=True)
ws.prepare(system, inp, exchange=exchange)
sim = UppASDSimulator(ws)
sim.initialize()
sim.measure()
sim.finalize()


## Inspect results

In [ ]:
res = ASDResults(workdir)

## Plot single spin trajectory

In [ ]:
traj = res['trajectory'][(1,1)]
plot_3d_trajectory(traj['mx'], traj['my'], traj['mz'], show_sphere=True)

## Load and plot spectrogram

In [ ]:
steps, spins = spin_snapshots(res, "moment")
# spins.shape == (Nt, Nsite, 3)
component = "z"   # or "x", "y"

comp_idx = {"x": 0, "y": 1, "z": 2}[component]

data_xt = spins[:, :, comp_idx]   # (Nt, Nsite)

plot_xt_map(
    data_xt,
    cmap="RdBu_r",
    xlabel="Site index",
    ylabel="Time step",
    title=fr"$m_{component}(x,t)$",
);
